In [1]:
import os
from pathlib import Path  
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

def connect_to_database(db_filename="database.sqlite"):
    """
    Zoekt dynamisch naar het databasebestand in de huidige werkmap of bovenliggende mappen
    en zet een veilige, foutbestendige SQLite-verbinding op.
    
    Args:
        db_filename (str): De exacte naam van het SQLite-bestand. Default is 'database.sqlite'.
        
    Returns:
        sqlite3.Connection: Een actieve databaseverbinding als het bestand bestaat.
        
    Raises:
        FileNotFoundError: Als het bestand nergens in de verwachte mappenstructuur wordt gevonden.
    """
    # Bepaal de huidige map waar het notebook draait 
    current_dir = Path.cwd()
    
    possible_paths = [
        current_dir / db_filename,
        current_dir / "notebook" / db_filename,
        current_dir.parent / "notebook" / db_filename,
        current_dir.parent / db_filename
    ]
    
    target_path = None
    for path in possible_paths:
        if path.is_file():
            target_path = path
            break
            
    # Harde fail-fast check om 'unsupported file format' errors door lege bestanden te voorkomen
    if not target_path:
        print("="*60)
        print("FOUT: Databasebestand kon niet automatisch worden gevonden!")
        print("Gecontroleerde locaties:")
        for path in possible_paths:
            print(f" - {path}")
        print("="*60)
        raise FileNotFoundError(f"Kon {db_filename} nergens vinden. Controleer de bestandsnaam.")
        
    # Bestand wel gevonden: log statistieken en maak de verbinding
    file_size_mb = target_path.stat().st_size / (1024 * 1024)
    print(f"✓ Database succesvol gelokaliseerd!")
    print(f"  Pad: {target_path}")
    print(f"  Grootte: {file_size_mb:.2f} MB")
    
    # Open de database in Read-Only modus (?mode=ro). 
    # Dit voorkomt dat SQLite stilletjes een leeg bestand aanmaakt bij een typefout.
    db_uri = f"file:{target_path}?mode=ro"
    connection = sqlite3.connect(db_uri, uri=True)
    print("✓ Succesvol verbonden met de database!")
    
    return connection

conn = connect_to_database()

✓ Database succesvol gelokaliseerd!
  Pad: c:\Users\sasha\Documents\GitHub\Datalab_semester2_Groep1\notebook\database.sqlite
  Grootte: 298.59 MB
✓ Succesvol verbonden met de database!
